# 03 — Machine-Learning Models for Nav1.7 Potency Prediction

## Overview

This notebook develops and evaluates quantitative structure–activity relationship (QSAR) models for predicting Nav1.7 inhibitory potency from molecular descriptors.

The analysis uses the 6,738 GOOD-quality compounds prepared and characterized in the previous notebooks. Experimental activity is represented as **pIC₅₀**, which serves as the continuous prediction target. Eleven molecular descriptors describing molecular size, lipophilicity, polarity, hydrogen bonding, flexibility, ring composition, and structural complexity are considered as candidate predictive features.

Three modeling approaches are evaluated:

1. **Linear QSAR** — provides an interpretable baseline for estimating the extent to which potency can be explained by linear combinations of molecular descriptors.
2. **Random Forest Regression** — captures nonlinear relationships and interactions among molecular properties using an ensemble of decision trees.
3. **XGBoost Regression** — uses gradient-boosted decision trees to model more complex structure–activity relationships.

Model performance is evaluated using a consistent validation framework and the same performance metrics across models, including **R², root mean squared error (RMSE), and mean absolute error (MAE)**. Cross-validation results are reported as **mean ± standard deviation** to assess both predictive performance and model stability.

Feature-importance analyses are used to examine which molecular descriptors contribute most strongly to predictive performance. Detailed interpretation of the final XGBoost model using SHAP is reserved for `04_model_interpretation.ipynb`.

## Modeling Objective

The objective of this notebook is to determine whether combining multiple molecular descriptors improves prediction of Nav1.7 inhibitory potency beyond the individual descriptor relationships observed during exploratory analysis, and to compare the performance of linear and nonlinear QSAR approaches.

## 1. Load Modeling Dataset

The modeling dataset exported from `02_exploratory_analysis.ipynb` is loaded for QSAR model development.

The dataset contains 6,738 GOOD-quality compounds, their experimental Nav1.7 activity values, data-quality metadata, and 11 molecular descriptors. Only the molecular descriptors are used as predictive features, while pIC₅₀ serves as the continuous prediction target.

In [9]:
import io
import numpy as np
import pandas as pd

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate

In [2]:
# Upload the modeling dataset exported from Notebook 02
uploaded = files.upload()

# Identify the uploaded file
file_name = list(uploaded.keys())[0]

# Load the dataset
modeling_df = pd.read_csv(
    io.BytesIO(uploaded[file_name])
)

print(f"Loaded: {file_name}")
print(
    f"Dataset dimensions: "
    f"{modeling_df.shape[0]:,} rows × {modeling_df.shape[1]} columns"
)

display(modeling_df.head())

Saving nav17_good_modeling_dataset.csv to nav17_good_modeling_dataset.csv
Loaded: nav17_good_modeling_dataset.csv
Dataset dimensions: 6,738 rows × 22 columns


,molecule_chembl_id,canonical_smiles,ic50_nm,pIC50,min_val,max_val,measurement_count,max_min_ratio,data_quality,confidence_level,...,LogP,TPSA,HBA,HBD,RotatableBonds,RingCount,AromaticRings,FractionCSP3,HeavyAtoms,Heteroatoms
0,CHEMBL3904041,C#Cc1cccc(OC2CN(c3ncnc(Nc4cccc(C(=O)NC)c4)n3)C...,990.0,6.004365,990.0,990.0,1,1.000000,GOOD,LOW,...,2.2237,92.27,7.0,2.0,6.0,4.0,3.0,0.181818,30.0,8.0
1,CHEMBL3662199,C/C=C/c1cc(C(=O)NS(=O)(=O)N2CCC2)c(F)cc1OCC12C...,6.0,8.221849,6.0,6.0,1,1.000000,GOOD,LOW,...,4.1343,75.71,4.0,1.0,7.0,6.0,1.0,0.625000,32.0,8.0
2,CHEMBL190461,C=C(C)[C@@H]1CCC(C)=C[C@H]1c1c(O)cc(CCCCC)cc1O,1820.0,5.739929,1820.0,1820.0,1,1.000000,GOOD,LOW,...,5.8465,40.46,2.0,2.0,6.0,2.0,1.0,0.523810,23.0,2.0
3,CHEMBL3955581,C=C(C)c1ccccc1N1CCOc2cc(S(=O)(=O)Nc3nccs3)ccc21,2420.0,5.616185,2420.0,2420.0,1,1.000000,GOOD,LOW,...,4.5075,71.53,6.0,1.0,5.0,4.0,3.0,0.150000,28.0,8.0
4,CHEMBL5805169,C=C(C)c1cn2c(NS(C)(=O)=O)nnc2cc1OCC12CC3CC(CC(...,51.1,7.291579,51.0,51.2,2,1.003922,GOOD,MEDIUM,...,3.7291,85.59,5.0,1.0,6.0,6.0,2.0,0.619048,29.0,8.0


In [3]:
# Validate the imported modeling dataset
print("--- Modeling Dataset Validation ---")

print(f"Compounds:            {len(modeling_df):,}")
print(f"Columns:              {len(modeling_df.columns)}")
print(f"Duplicate structures: {modeling_df['canonical_smiles'].duplicated().sum():,}")
print(f"Missing values:       {modeling_df.isna().sum().sum():,}")

print("\nColumns:")
print(modeling_df.columns.tolist())

--- Modeling Dataset Validation ---
Compounds:            6,738
Columns:              22
Duplicate structures: 0
Missing values:       0

Columns:
['molecule_chembl_id', 'canonical_smiles', 'ic50_nm', 'pIC50', 'min_val', 'max_val', 'measurement_count', 'max_min_ratio', 'data_quality', 'confidence_level', 'potency_class', 'MW', 'LogP', 'TPSA', 'HBA', 'HBD', 'RotatableBonds', 'RingCount', 'AromaticRings', 'FractionCSP3', 'HeavyAtoms', 'Heteroatoms']


## 2. Modeling Features and Prediction Target

Nav1.7 inhibitory potency, expressed as **pIC₅₀**, is used as the continuous prediction target.

Eleven molecular descriptors calculated in the previous notebook are considered as candidate predictive features. Compound identifiers, SMILES strings, activity-derived variables, potency classifications, and data-quality metadata are excluded from the feature matrix to prevent target leakage.

A fixed 80/20 train–test split is created before model fitting or model-specific feature selection. The training set is used for model development and cross-validation, while the held-out test set is reserved for final evaluation.

In [4]:
# Define prediction target
target = 'pIC50'

# Define candidate molecular descriptors
candidate_features = [
    'MW',
    'LogP',
    'TPSA',
    'HBA',
    'HBD',
    'RotatableBonds',
    'RingCount',
    'AromaticRings',
    'FractionCSP3',
    'HeavyAtoms',
    'Heteroatoms'
]

# Construct feature matrix and target vector
X = modeling_df[candidate_features].copy()
y = modeling_df[target].copy()

print(f"Feature matrix: {X.shape[0]:,} compounds × {X.shape[1]} descriptors")
print(f"Target: {target}")

Feature matrix: 6,738 compounds × 11 descriptors
Target: pIC50


## 2. Modeling Dataset and Validation Strategy

Nav1.7 inhibitory potency, expressed as **pIC₅₀**, is used as the continuous prediction target. Eleven molecular descriptors are initially considered as candidate predictive features:

- MW
- LogP
- TPSA
- HBA
- HBD
- RotatableBonds
- RingCount
- AromaticRings
- FractionCSP3
- HeavyAtoms
- Heteroatoms

Compound identifiers, SMILES strings, activity-derived variables, potency classifications, and data-quality metadata are excluded from the predictive feature matrix to prevent target leakage.

The 6,738-compound dataset is divided into **80% training data and 20% held-out test data** using a fixed random seed for reproducibility. The same train–test split is retained for all models evaluated in this notebook.

Model development and cross-validation are performed using only the training set. Five-fold cross-validation is used to estimate predictive performance and model stability during development. The held-out test set is reserved for final evaluation and is not used for feature selection, model fitting, or cross-validation.

Model performance is evaluated using **R²**, **root mean squared error (RMSE)**, and **mean absolute error (MAE)**.

In [6]:
# Create a reproducible 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("--- Train/Test Split ---")
print(f"Training compounds: {len(X_train):,} ({len(X_train) / len(X):.1%})")
print(f"Test compounds:     {len(X_test):,} ({len(X_test) / len(X):.1%})")

--- Train/Test Split ---
Training compounds: 5,390 (80.0%)
Test compounds:     1,348 (20.0%)


## 3. Reduced Linear QSAR Model

A multiple linear regression model is first developed as an interpretable QSAR baseline for predicting Nav1.7 inhibitory potency.

Unlike the nonlinear models evaluated later, linear regression is sensitive to multicollinearity among predictors. The exploratory analysis identified substantial correlations and elevated Variance Inflation Factors (VIFs) among several of the 11 molecular descriptors, particularly descriptors representing related aspects of molecular size, polarity, and ring composition.

### 3.1 Chemistry-Informed Descriptor Reduction

To reduce redundancy while preserving chemically distinct information, a representative descriptor was selected for each major physicochemical concept:

- **MW:** molecular size
- **LogP:** lipophilicity
- **TPSA:** molecular polarity
- **HBD:** hydrogen-bond donation
- **RotatableBonds:** molecular flexibility
- **RingCount:** ring topology
- **FractionCSP3:** saturation and three-dimensional character

Four descriptors were excluded from the reduced linear model because they substantially overlap with retained descriptors representing related chemical information:

- **HeavyAtoms:** strongly associated with molecular size and highly correlated with MW
- **Heteroatoms:** overlaps with molecular size, polarity, and heteroatom-related descriptors
- **HBA:** substantially associated with TPSA and other measures of molecular polarity
- **AromaticRings:** overlaps with ring composition and is strongly related to FractionCSP3 in this dataset

This reduction is intended to improve the stability and interpretability of the linear regression coefficients rather than to identify a universally optimal feature subset. The excluded descriptors are therefore not considered chemically unimportant and remain available to the nonlinear Random Forest and XGBoost models.

To preserve the independence of the held-out test set, multicollinearity of the reduced feature set is evaluated using only the training data. VIF values are recalculated after descriptor reduction to determine the remaining degree of redundancy among the seven retained predictors.

In [7]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# Define chemistry-informed reduced descriptor set
reduced_features = [
    'MW',
    'LogP',
    'TPSA',
    'HBD',
    'RotatableBonds',
    'RingCount',
    'FractionCSP3'
]

# Select reduced descriptors from the training set only
X_train_reduced = X_train[reduced_features].copy()
X_test_reduced = X_test[reduced_features].copy()

# Add intercept temporarily for VIF calculation
X_vif = sm.add_constant(X_train_reduced)

# Calculate VIF for each retained descriptor
reduced_vif = pd.DataFrame({
    'Descriptor': reduced_features,
    'VIF': [
        variance_inflation_factor(X_vif.values, i)
        for i in range(1, X_vif.shape[1])
    ]
})

# Sort from highest to lowest VIF
reduced_vif = reduced_vif.sort_values(
    'VIF',
    ascending=False
).reset_index(drop=True)

print("--- VIF After Chemistry-Informed Descriptor Reduction ---")
display(reduced_vif.round(2))

--- VIF After Chemistry-Informed Descriptor Reduction ---


,Descriptor,VIF
0,MW,3.51
1,TPSA,3.46
2,LogP,3.21
3,FractionCSP3,2.09
4,RotatableBonds,1.71
5,HBD,1.67
6,RingCount,1.33


### Reduced-Set Multicollinearity Assessment

After chemistry-informed descriptor reduction, all seven retained descriptors showed VIF values below 5, with the highest values observed for MW (3.51), TPSA (3.46), and LogP (3.21).

This represents a substantial reduction in multicollinearity compared with the original 11-descriptor feature space, in which MW and HeavyAtoms had VIF values above 10. The remaining VIF values indicate that the reduced descriptors retain related chemical information without exhibiting severe multicollinearity.

The seven-descriptor feature set was therefore retained for the linear QSAR model without further reduction.

### 3.2 Reduced Linear QSAR Equation

A multiple linear regression model is fitted using the seven retained molecular descriptors and the compounds in the training set. Nav1.7 inhibitory potency, expressed as **pIC₅₀**, is used as the continuous response variable.

The model takes the general form:

**pIC₅₀ = β₀ + β₁(MW) + β₂(LogP) + β₃(TPSA) + β₄(HBD) + β₅(RotatableBonds) + β₆(RingCount) + β₇(FractionCSP3)**

where β₀ represents the intercept and each β coefficient represents the estimated contribution of its corresponding molecular descriptor while the remaining descriptors are held constant.

The model is fitted exclusively on the training set. Predictive generalizability is evaluated separately using five-fold cross-validation in the following section.

In [8]:
# Add intercept for statsmodels linear regression
X_train_sm = sm.add_constant(X_train_reduced)

# Fit reduced linear QSAR using training data only
linear_model = sm.OLS(
    y_train,
    X_train_sm
).fit()

# Extract fitted coefficients
params = linear_model.params

# Construct readable QSAR equation
equation_terms = [f"{params['const']:.3f}"]

for feature in reduced_features:
    coefficient = params[feature]
    sign = "+" if coefficient >= 0 else "-"

    equation_terms.append(
        f"{sign} {abs(coefficient):.4f}({feature})"
    )

print("--- Reduced Linear QSAR Equation ---")
print("pIC₅₀ = " + " ".join(equation_terms))

print("\n--- Linear Regression Summary ---")
print(linear_model.summary())

--- Reduced Linear QSAR Equation ---
pIC₅₀ = 1.987 + 0.0046(MW) + 0.3082(LogP) + 0.0048(TPSA) + 0.1779(HBD) - 0.0409(RotatableBonds) + 0.0582(RingCount) + 1.4670(FractionCSP3)

--- Linear Regression Summary ---
                            OLS Regression Results                            
Dep. Variable:                  pIC50   R-squared:                       0.270
Model:                            OLS   Adj. R-squared:                  0.269
Method:                 Least Squares   F-statistic:                     284.9
Date:                Mon, 27 Jul 2026   Prob (F-statistic):               0.00
Time:                        02:01:38   Log-Likelihood:                -6933.8
No. Observations:                5390   AIC:                         1.388e+04
Df Residuals:                    5382   BIC:                         1.394e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         

#### Initial Model Fit

The fitted reduced linear QSAR model produced a training-set **R² of 0.270** and an adjusted **R² of 0.269**, indicating that the seven retained descriptors collectively explain approximately 27% of the variation in pIC₅₀ within the training data.

Positive coefficients were obtained for MW, LogP, TPSA, HBD, RingCount, and FractionCSP3, whereas RotatableBonds showed a negative coefficient. These coefficients represent conditional associations within the multivariable model and should not be interpreted as independent causal effects of individual molecular properties.

The moderate training-set R² indicates that the linear model captures meaningful but incomplete information about Nav1.7 inhibitory potency. Its predictive stability and generalizability are evaluated next using five-fold cross-validation.

### 3.3 Five-Fold Cross-Validation

The training set is evaluated using **five-fold cross-validation** to assess the predictive performance and stability of the reduced linear QSAR model beyond its fit to the full training data.

The 5,390 training compounds are randomly divided into five folds using a fixed random seed for reproducibility. In each iteration, the model is fitted on four folds and evaluated on the remaining fold. This process is repeated five times so that each compound serves once as validation data.

Performance is evaluated using three complementary regression metrics:

- **R²:** proportion of variation in pIC₅₀ explained by the model
- **RMSE:** root mean squared prediction error, which gives greater weight to larger errors
- **MAE:** mean absolute prediction error

Results are reported as **mean ± standard deviation** across the five validation folds.

The held-out test set is not used during cross-validation and remains reserved for final model evaluation.

In [10]:
# Define reproducible 5-fold cross-validation
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Define the reduced linear QSAR model
linear_cv_model = LinearRegression()

# Perform cross-validation using the training set only
linear_cv_results = cross_validate(
    linear_cv_model,
    X_train_reduced,
    y_train,
    cv=cv,
    scoring={
        'r2': 'r2',
        'rmse': 'neg_root_mean_squared_error',
        'mae': 'neg_mean_absolute_error'
    }
)

# Extract validation scores
linear_cv_r2 = linear_cv_results['test_r2']
linear_cv_rmse = -linear_cv_results['test_rmse']
linear_cv_mae = -linear_cv_results['test_mae']

# Report individual fold results
linear_cv_summary = pd.DataFrame({
    'Fold': range(1, 6),
    'R²': linear_cv_r2,
    'RMSE': linear_cv_rmse,
    'MAE': linear_cv_mae
})

print("--- Reduced Linear QSAR: 5-Fold Cross-Validation ---")
display(linear_cv_summary.round(3))

print("--- Mean ± SD Cross-Validation Performance ---")
print(f"R²:   {linear_cv_r2.mean():.3f} ± {linear_cv_r2.std():.3f}")
print(f"RMSE: {linear_cv_rmse.mean():.3f} ± {linear_cv_rmse.std():.3f} pIC₅₀ units")
print(f"MAE:  {linear_cv_mae.mean():.3f} ± {linear_cv_mae.std():.3f} pIC₅₀ units")

--- Reduced Linear QSAR: 5-Fold Cross-Validation ---


,Fold,R²,RMSE,MAE
0,1,0.268,0.878,0.708
1,2,0.261,0.876,0.706
2,3,0.275,0.880,0.702
3,4,0.271,0.878,0.702
4,5,0.261,0.873,0.695


--- Mean ± SD Cross-Validation Performance ---
R²:   0.267 ± 0.006
RMSE: 0.877 ± 0.003 pIC₅₀ units
MAE:  0.703 ± 0.005 pIC₅₀ units


#### Cross-Validation Results

Five-fold cross-validation produced a mean **R² of 0.267 ± 0.006**, **RMSE of 0.877 ± 0.003 pIC₅₀ units**, and **MAE of 0.703 ± 0.005 pIC₅₀ units**.

Performance was highly consistent across the five validation folds. R² values ranged from 0.261 to 0.275, while RMSE and MAE varied only slightly between folds. The cross-validated mean R² was also close to the full training-set R² of 0.270, indicating limited evidence of overfitting within the reduced linear model.

However, the cross-validated R² of approximately 0.27 indicates that most of the observed variation in Nav1.7 inhibitory potency is not captured by a simple linear combination of the seven selected molecular descriptors. This provides a baseline against which the nonlinear Random Forest and XGBoost models can be evaluated.

## 4. Random Forest QSAR Model

The reduced linear QSAR model provides an interpretable baseline but assumes that descriptor effects are additive and approximately linear. Molecular structure–activity relationships may instead involve nonlinear effects and interactions among physicochemical properties.

A **Random Forest regression model** is therefore evaluated as a nonlinear QSAR approach. Random Forest combines predictions from an ensemble of decision trees and can capture nonlinear relationships and interactions without requiring an explicit mathematical relationship between descriptors and potency.

Unlike the reduced linear QSAR model, the Random Forest model uses the **full set of 11 molecular descriptors**. The chemistry-informed descriptor reduction performed previously was intended specifically to reduce multicollinearity and improve coefficient interpretation in linear regression. Tree-based models are less directly affected by predictor multicollinearity, so potentially informative descriptors are retained for nonlinear modeling.

The same training set and five-fold cross-validation framework established previously are used to allow direct comparison with the linear QSAR model. The held-out test set remains unused until final model evaluation.

### 4.1 Random Forest Cross-Validation

Random Forest regression is evaluated using the same five cross-validation folds used for the reduced linear QSAR model.

Performance is assessed using **R², RMSE, and MAE**, with results reported as mean ± standard deviation across the five validation folds. Using the same training compounds, validation folds, and evaluation metrics provides a consistent basis for comparing linear and nonlinear QSAR performance.

In [11]:
from sklearn.ensemble import RandomForestRegressor

# Define Random Forest model
rf_cv_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Perform 5-fold cross-validation using all 11 descriptors
rf_cv_results = cross_validate(
    rf_cv_model,
    X_train,
    y_train,
    cv=cv,
    scoring={
        'r2': 'r2',
        'rmse': 'neg_root_mean_squared_error',
        'mae': 'neg_mean_absolute_error'
    },
    n_jobs=-1
)

# Extract validation scores
rf_cv_r2 = rf_cv_results['test_r2']
rf_cv_rmse = -rf_cv_results['test_rmse']
rf_cv_mae = -rf_cv_results['test_mae']

# Build fold-by-fold summary
rf_cv_summary = pd.DataFrame({
    'Fold': range(1, 6),
    'R²': rf_cv_r2,
    'RMSE': rf_cv_rmse,
    'MAE': rf_cv_mae
})

print("--- Random Forest QSAR: 5-Fold Cross-Validation ---")
display(rf_cv_summary.round(3))

print("--- Mean ± SD Cross-Validation Performance ---")
print(f"R²:   {rf_cv_r2.mean():.3f} ± {rf_cv_r2.std():.3f}")
print(f"RMSE: {rf_cv_rmse.mean():.3f} ± {rf_cv_rmse.std():.3f} pIC₅₀ units")
print(f"MAE:  {rf_cv_mae.mean():.3f} ± {rf_cv_mae.std():.3f} pIC₅₀ units")

--- Random Forest QSAR: 5-Fold Cross-Validation ---


,Fold,R²,RMSE,MAE
0,1,0.555,0.684,0.519
1,2,0.522,0.705,0.523
2,3,0.531,0.708,0.528
3,4,0.562,0.681,0.508
4,5,0.522,0.702,0.517


--- Mean ± SD Cross-Validation Performance ---
R²:   0.538 ± 0.017
RMSE: 0.696 ± 0.011 pIC₅₀ units
MAE:  0.519 ± 0.007 pIC₅₀ units


In [12]:
# Fit Random Forest on the complete training set
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print(
    f"Random Forest fitted on "
    f"{len(X_train):,} training compounds using "
    f"{X_train.shape[1]} molecular descriptors."
)

Random Forest fitted on 5,390 training compounds using 11 molecular descriptors.


#### Random Forest Cross-Validation Results

The Random Forest model substantially outperformed the reduced linear QSAR model during five-fold cross-validation. It achieved a mean **R² of 0.538 ± 0.017**, **RMSE of 0.696 ± 0.011 pIC₅₀ units**, and **MAE of 0.519 ± 0.007 pIC₅₀ units**.

Performance was reasonably consistent across the five folds, with R² values ranging from 0.522 to 0.562. Compared with the reduced linear QSAR model (R² = 0.267 ± 0.006), the Random Forest model explained substantially more of the variation in Nav1.7 inhibitory potency while producing lower prediction errors.

The improvement suggests that relationships between the molecular descriptors and Nav1.7 potency are not adequately represented by a purely additive linear model. Nonlinear descriptor effects and interactions appear to contribute meaningful predictive information.

These results establish Random Forest as a stronger nonlinear benchmark for subsequent comparison with XGBoost. The held-out test set remains reserved for final model evaluation.